# Demo 09g — Imputation with NMF: ratings, time series, topics and the start

**Class 09 · E4** (slides: `slides/09_blind_signal_separation.md`, blocks "Collaborative Filtering", "Topics" and "Time Series")

Filling a matrix with holes is one problem with three faces: **collaborative filtering** (users x items, 85 % of the
ratings unknown), **time series** (series x hours, blocks of consecutive hours lost) and **text** (documents x terms,
where the "hole" is the word a document could have used). NMF with a mask fits only the observed cells; the question of
every section is where it beats a simple baseline and where it does not. The last section tests the *start* of the
non-convex problem.

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import pynnmf
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer

from faa import bss, data, nmfinit, plotlib

warnings.filterwarnings("ignore")
pl.Config.set_tbl_rows(40)
pl.Config.set_tbl_cols(20)
pl.Config.set_float_precision(3)
pl.Config.set_tbl_hide_dataframe_shape(True)
pl.Config.set_tbl_formatting("ASCII_MARKDOWN")
pl.Config.set_tbl_width_chars(150)
plotlib.setup(module="09")
BLUE, ORANGE, GREEN, RED, SKY, PURPLE, GREY = (
    plotlib.BLUE,
    plotlib.ORANGE,
    plotlib.GREEN,
    plotlib.RED,
    plotlib.SKY,
    plotlib.PURPLE,
    plotlib.GREY,
)
SEED = 0


def slide(title: str) -> None:
    """Every cell that prints a number of a slide starts with this line."""
    print(f"--- slide: {title} ---")

## Slide: Collaborative filtering — the data and the baselines

**MovieLens 100K** (GroupLens): 943 users, 1682 films, 100 000 ratings of 1-5 stars. We keep the 603 films with at least 50
ratings (14.7 % of the matrix observed). 20 % of the observed ratings are held out as the test set, and 30 users lose *all*
their training ratings (cold start). Before any factorisation: the global mean, the user mean, the item mean and the **bias
model** (mean + item bias + user bias), the baseline a factorisation must beat.

In [ ]:
R_full, obs_full, titles = data.load_movielens()
keep = obs_full.sum(0) >= 50
R, obs, titles = R_full[:, keep], obs_full[:, keep], [t for t, k in zip(titles, keep) if k]
train, test = bss.holdout(obs, 0.2, seed=SEED)
cold = np.random.default_rng(SEED).choice(R.shape[0], 30, replace=False)
train[cold] = False
is_cold = np.zeros(R.shape[0], dtype=bool)
is_cold[cold] = True
test_cold = test & is_cold[:, None]  # ratings of the cold users, scored apart
test = test & ~is_cold[:, None]
Rn = np.nan_to_num(R)
Rtr = np.where(train, R, np.nan)
rmse = lambda P, m=test: float(np.sqrt(((P - Rn)[m] ** 2).mean()))  # noqa: E731
mu = float(np.nanmean(Rtr))
im = np.nanmean(Rtr, 0)
im = np.where(np.isnan(im), mu, im)
um = np.nanmean(Rtr, 1)
um = np.where(np.isnan(um), mu, um)
Bm = bss.bias_baseline(R, train)
base = {
    "global mean": rmse(np.full(R.shape, mu)),
    "user mean": rmse(np.tile(um[:, None], (1, R.shape[1]))),
    "item mean": rmse(np.tile(im, (R.shape[0], 1))),
    "bias model": rmse(Bm),
}
slide("Collaborative filtering: data and baselines")
print(
    f"{R.shape[0]} users x {R.shape[1]} films, {obs.mean():.1%} observed, train {int(train.sum())}, test {int(test.sum())} ratings of warm users, {len(cold)} cold-start users with {int(test_cold.sum())} test ratings"
)
print({k: round(v, 3) for k, v in base.items()})

## Slide: Collaborative filtering — rank and regularisation

`rwnmf` (weighted NMF, mask-aware) over the rank c and the L2 weight alpha. Training error always falls with c; the
held-out error falls and then rises. Regularisation moves the best rank up. The choice is made on a validation split of the
*training* ratings; the test ratings are used once, at the end.

In [ ]:
tr2, val = bss.holdout(train, 0.2, seed=1)
Rtr2 = np.where(tr2, R, np.nan)
rows = []
for c in (1, 2, 3, 4, 6, 8):
    for a in (0.3, 1.0, 3.0, 10.0):
        q = pynnmf.rwnmf(Rtr2, c, alpha=a, num_iter=200, seed=SEED, init="random")
        P = q[1] @ q[2].T
        rows.append({"c": c, "alpha": a, "validation": rmse(P, val), "train": rmse(P, tr2)})
grid = pl.DataFrame(rows)
best = grid.sort("validation").row(0, named=True)
q = pynnmf.rwnmf(Rtr, best["c"], alpha=best["alpha"], num_iter=200, seed=SEED, init="random")
P_best = q[1] @ q[2].T
slide("Collaborative filtering: rank and regularisation sweep")
print(grid.pivot(on="alpha", index="c", values="validation").with_columns(pl.col(pl.Float64).round(3)))
print(
    f"validation picks c = {best['c']}, alpha = {best['alpha']}; test RMSE {rmse(P_best):.3f} (train {rmse(P_best, train):.3f}); bias model {base['bias model']:.3f}; item mean {base['item mean']:.3f}"
)
weak = grid.filter((pl.col("alpha") == 0.3) & (pl.col("c") == 8)).row(0, named=True)
print(f"c = 8 with little regularisation: train {weak['train']:.3f}, validation {weak['validation']:.3f}")
assert rmse(P_best) < base["bias model"] < base["item mean"], (
    "NMF must beat the bias model once rank and alpha are validated"
)
assert weak["validation"] > 1.15 * weak["train"] + 0.1 and best["c"] <= 4

f, ax = plt.subplots(1, 2, figsize=(plotlib.FULL, 1.5), sharey=True)
for a_, col in ((0.3, RED), (3.0, GREEN)):
    g = grid.filter(pl.col("alpha") == a_)
    ax[0].plot(g["c"], g["train"], color=col, ls="--", lw=1.0)
    ax[0].plot(g["c"], g["validation"], color=col, lw=1.4, label=f"alpha {a_}")
ax[0].axhline(base["bias model"], color=GREY, ls=":", lw=1)
ax[0].text(5.5, base["bias model"] - 0.035, "bias model", ha="center", fontsize=6, color=GREY)
ax[0].set_ylim(0.7, 1.2)
ax[0].set_xlabel("rank c"), ax[0].set_ylabel("RMSE"), ax[0].set_title("dashed: train, solid: validation", fontsize=6.5)
ax[0].legend(fontsize=6, loc="upper left", ncol=2)
vals_ = list(base.values()) + [rmse(P_best)]
ax[1].scatter(range(5), vals_, color=[GREY, GREY, GREY, BLUE, ORANGE], s=18, zorder=3)
ax[1].vlines(range(5), 0.7, vals_, color="#C8C8C8", lw=1.0)
for k_, v_ in enumerate(vals_):
    ax[1].text(k_, v_ + 0.025, f"{v_:.3f}", ha="center", fontsize=5.5)
ax[1].set_ylim(0.7, 1.2)
ax[1].set_xticks(range(5)), ax[1].set_xticklabels(["global", "user", "item", "bias", "NMF"], fontsize=6)
ax[1].set_title("test RMSE", fontsize=6.5)
plotlib.save(f, "p7_cf")

## Slide: Collaborative filtering — every solver, cold start, top-N

The four pyNNMF solvers at the validated rank (alpha only exists in `rwnmf`; the others have none). The 30 cold-start users
have no training rating: their row of W has nothing to fit, so the model falls back on zero. Top-N: hide the liked films
(>= 4 stars) of each user and rank all unseen films; the hit rate at 10 is the share of users with a liked film in their
top 10.

In [ ]:
c_ = best["c"]
solvers = {
    "rwnmf": lambda: (lambda q: q[1] @ q[2].T)(
        pynnmf.rwnmf(Rtr, c_, alpha=best["alpha"], num_iter=200, seed=SEED, init="random")
    ),
    "HALS": lambda: (lambda q: q[1] @ q[2])(pynnmf.nmf_hals(Rtr, c_, n=200, seed=SEED, init="random")),
    "MU": lambda: (lambda q: q[1] @ q[2])(pynnmf.nmf_mu(Rtr, c_, n=200, seed=SEED, init="random")),
    "ALS": lambda: (lambda q: q[1] @ q[2])(pynnmf.nmf_als(Rtr, c_, n=200, seed=SEED, init="random")),
}
preds = {k: f() for k, f in solvers.items()}
slide("Collaborative filtering: solvers at the validated rank")
print({k: round(rmse(P), 3) for k, P in preds.items()}, "test RMSE; bias model", round(base["bias model"], 3))

rows = []
for name, m in {"cold (0 ratings)": test_cold, "warm": test}.items():
    rows.append(
        {
            "users": name,
            "test cells": int(m.sum()),
            "item mean": rmse(np.tile(im, (R.shape[0], 1)), m),
            "bias model": rmse(Bm, m),
            "NMF": rmse(P_best, m),
        }
    )
slide("Collaborative filtering: error of cold and warm users")
print(pl.DataFrame(rows).with_columns(pl.col(pl.Float64).round(3)))
print(f"cold users get {np.mean(P_best[is_cold]):.2f} on average from NMF (stars are 1-5)")


def hit_rate(P: np.ndarray, k: int = 10) -> float:
    hits, users = 0, 0
    for i in range(R.shape[0]):
        liked = test[i] & (Rn[i] >= 4)
        if not liked.any():
            continue
        score = np.where(train[i], -np.inf, P[i])
        hits += bool(np.isin(np.argsort(score)[::-1][:k], np.flatnonzero(liked)).any())
        users += 1
    return hits / users


pop = np.tile(train.sum(0).astype(float), (R.shape[0], 1))
hr = {"popularity": hit_rate(pop), "bias model": hit_rate(Bm), "NMF": hit_rate(P_best)}
slide("Collaborative filtering: hit rate at 10")
print({k: round(v, 3) for k, v in hr.items()})
topf = np.argsort(P_best[~is_cold].mean(0))[::-1][:5]
print("films with the highest mean predicted rating:", [titles[j] for j in topf])

## Slide: Collaborative filtering — what the factors mean

Each column of H is a taste: the films that weigh most in it. NMF factors are non-negative, so a taste is a list of films,
not a contrast.

In [ ]:
q = pynnmf.rwnmf(Rtr, c_, alpha=best["alpha"], num_iter=200, seed=SEED, init="random")
Hf = q[2].T  # (c, films) after the (n, c) convention of rwnmf
slide("Collaborative filtering: the films of each taste")
for k in range(c_):
    print(f"taste {k + 1}:", "; ".join(titles[j] for j in np.argsort(Hf[k])[::-1][:4]))

## Slide: Time series — blocks of consecutive hours are lost

The room temperatures of one house (UCI *Appliances energy prediction*: 8 rooms, hourly means, 60 days = 1440 hours). Four
blocks of 48 hours per room are hidden and scored; three more blocks of 72 hours are missing for the fit but not scored,
as sensor failures are. Baselines: linear interpolation (univariate) and
iterative PCA (EM). NMF is `nmf_hals`/`rwnmf` with the mask. **Gap alignment** (Fernandes et al., Machine Learning 2021)
shifts every imputed block so its ends meet the observed neighbours.

In [ ]:
Xall, names = data.load_indoor_climate()
X = Xall[[i for i, n in enumerate(names) if n.startswith("T")]][:, :1440]  # the eight room temperatures, first 60 days
N_S, T_S = X.shape


def ts_case(seed: int, length: int = 48):
    held = bss.ts_gaps(X.shape, seed, length=length)
    pre = bss.ts_gaps(X.shape, seed + 50, length=72, per_series=3) & ~held
    return held, ~(held | pre)


def fill_nmf(name: str, Xobs: np.ndarray, c: int, init, seed: int) -> np.ndarray:
    if name == "HALS":
        q = pynnmf.nmf_hals(Xobs, c, n=300, seed=seed, init=init)
        return q[1] @ q[2]
    q = pynnmf.rwnmf(Xobs, c, num_iter=300, seed=seed, init=init)
    return q[1] @ q[2].T


rows = []
for seed in range(5):
    held, ob = ts_case(seed)
    Xobs = np.where(ob, X, np.nan)
    Xz = np.where(ob, X, 0.0)
    score = lambda Y: float(np.sqrt(((Y - X)[held] ** 2).mean()))  # noqa: E731
    rows.append({"seed": seed, "method": "linear", "RMSE": score(bss.linear_fill(Xz, ob))})
    rows.append({"seed": seed, "method": "iterative PCA c=2", "RMSE": score(bss.pca_em(Xz, ob, 2))})
    for c in (1, 2, 3, 4):
        for name in ("HALS", "rwnmf"):
            for init in ("random", "svd_impute"):
                Rh = [fill_nmf(name, Xobs, c, init, s) for s in range(3 if init == "random" else 1)]
                rows.append(
                    {
                        "seed": seed,
                        "method": f"{name} c={c} {init}",
                        "RMSE": float(np.mean([score(np.where(ob, X, r)) for r in Rh])),
                    }
                )
                rows.append(
                    {
                        "seed": seed,
                        "method": f"{name} c={c} {init} +align",
                        "RMSE": float(np.mean([score(bss.align_gaps(Xz, ob, np.where(ob, X, r))) for r in Rh])),
                    }
                )
ts = (
    pl.DataFrame(rows)
    .group_by("method", maintain_order=True)
    .agg(pl.col("RMSE").mean().alias("RMSE"), pl.col("RMSE").std().alias("std"))
)
get = lambda m: float(ts.filter(pl.col("method") == m)["RMSE"][0])  # noqa: E731
slide("Time series: RMSE on the hidden 48-hour blocks (5 datasets)")
print(
    ts.filter(
        pl.col("method").is_in(
            ["linear", "iterative PCA c=2"]
            + [f"HALS c={c} {i}{a}" for c in (1, 2, 3, 4) for i in ("random", "svd_impute") for a in ("", " +align")]
        )
    ).with_columns(pl.col(pl.Float64).round(3))
)
print(
    ts.filter(pl.col("method").str.starts_with("rwnmf") & pl.col("method").str.contains("random")).with_columns(
        pl.col(pl.Float64).round(3)
    )
)
assert get("HALS c=1 random +align") < get("HALS c=1 random") < get("linear"), (
    "rank 1 beats linear, alignment adds more"
)
assert all(get(f"HALS c={c} random") > get("HALS c=2 random") for c in (3, 4)) and get("HALS c=4 random") > get(
    "linear"
), "high rank overfits the gaps"

held, ob = ts_case(0)
Xobs = np.where(ob, X, np.nan)
Xz = np.where(ob, X, 0.0)
bad = np.where(ob, X, fill_nmf("HALS", Xobs, 3, "svd_impute", 0))
fixed = bss.align_gaps(Xz, ob, bad)
good = np.where(ob, X, fill_nmf("HALS", Xobs, 2, "random", 0))
lin = bss.linear_fill(Xz, ob)
i = int(np.argmax([(held[k] & ~ob[k]).sum() for k in range(N_S)]))
a = int(np.flatnonzero(held[i])[0])
sl = slice(max(a - 60, 0), a + 48 + 60)
f, ax = plt.subplots(figsize=(plotlib.FULL, 1.6))
tt = np.arange(T_S)[sl]
ax.axvspan(a, a + 47, color="#EEF0F3", lw=0)
ax.plot(tt, X[i, sl], color=GREY, lw=1.6, label="truth")
ax.plot(tt, np.where(ob, X, np.nan)[i, sl], color="black", lw=0.9, label="observed")
ax.plot(tt, lin[i, sl], color=SKY, lw=0.9, ls=":", label="linear")
ax.plot(tt, bad[i, sl], color=RED, lw=1.0, label="NMF, SVD start")
ax.plot(tt, fixed[i, sl], color=ORANGE, lw=1.1, label="+ alignment")
ax.plot(tt, good[i, sl], color=GREEN, lw=1.0, ls="--", label="NMF, random start")
ax.set_xlabel("hour"), ax.set_ylabel("value")
ax.legend(fontsize=5.5, ncol=3, loc="upper center", bbox_to_anchor=(0.5, 1.28))
plotlib.save(f, "p7_ts_gap")

f, ax = plt.subplots(figsize=(plotlib.COL, 1.5))
cs = (1, 2, 3, 4)
for init, col in (("random", GREEN), ("svd_impute", RED)):
    ax.plot(
        cs,
        [get(f"HALS c={c} {init}") for c in cs],
        color=col,
        lw=1.4,
        marker="o",
        ms=2.5,
        label=f"HALS, {init.replace('_', ' ')}",
    )
    ax.plot(
        cs,
        [get(f"HALS c={c} {init} +align") for c in cs],
        color=col,
        lw=1.0,
        ls="--",
        label="+ alignment" if init == "random" else None,
    )
ax.axhline(get("linear"), color=SKY, ls=":", lw=1)
ax.text(4, get("linear") - 0.12, "linear", ha="right", fontsize=6, color=SKY)
ax.axhline(get("iterative PCA c=2"), color=BLUE, ls=":", lw=1)
ax.text(4, get("iterative PCA c=2") + 0.05, "iterative PCA", ha="right", fontsize=6, color=BLUE)
ax.set_yscale("log"), ax.set_xticks(cs), ax.set_xlabel("rank c"), ax.set_ylabel("RMSE (log)")
ax.legend(fontsize=5.5, loc="upper center", bbox_to_anchor=(0.5, -0.4), ncol=2)
plotlib.save(f, "p7_ts_rank")

## Slide: Time series — how long a gap can NMF bridge?

Gap length 12 to 96 hours, HALS c = 2 with the random start, with and without alignment, against linear interpolation.

In [ ]:
rows = []
for length in (12, 24, 48, 96):
    r = {"gap (h)": length}
    for seed in range(3):
        held_, ob_ = ts_case(seed, length)
        Xo, Xz_ = np.where(ob_, X, np.nan), np.where(ob_, X, 0.0)
        sc = lambda Y: float(np.sqrt(((Y - X)[held_] ** 2).mean()))  # noqa: E731
        Rh = np.where(ob_, X, fill_nmf("HALS", Xo, 2, "random", seed))
        for k, v in (
            ("linear", sc(bss.linear_fill(Xz_, ob_))),
            ("HALS c=2", sc(Rh)),
            ("HALS c=2 + align", sc(bss.align_gaps(Xz_, ob_, Rh))),
        ):
            r[k] = r.get(k, 0.0) + v / 3
    rows.append(r)
slide("Time series: RMSE by gap length")
gl = pl.DataFrame(rows)
print(gl.with_columns(pl.col(pl.Float64).round(3)))
short, long_ = (
    gl.filter(pl.col("gap (h)") == 12).row(0, named=True),
    gl.filter(pl.col("gap (h)") == 96).row(0, named=True),
)
assert short["linear"] < short["HALS c=2"] and short["HALS c=2 + align"] < short["linear"], (
    "short gaps: linear beats raw NMF, aligned NMF beats linear"
)
assert long_["HALS c=2 + align"] > long_["HALS c=2"] - 0.01, "long gaps: alignment no longer helps"

## Slide: Topics — latent semantic indexing and NMF on the same documents

**BBC news**: 1000 articles, 200 each of business, entertainment, politics, sport and tech (Greene and Cunningham 2006).
tf-idf with 3000 terms. LSI is a truncated SVD of the tf-idf matrix: signed axes, good for search. NMF with the same c = 5
gives non-negative topics: word lists. A query is projected into the latent space and compared there, so a document can
match a query it shares no word with. The categories are never used by the models, only to score them.

In [ ]:
texts, cat_labels = data.load_news()
labs = np.array(cat_labels)
cat_names = sorted(set(cat_labels))
vec = TfidfVectorizer(stop_words="english", min_df=10, max_df=0.5, max_features=3000)
Xt = vec.fit_transform(texts).toarray()
vocab = np.array(vec.get_feature_names_out())
svd = TruncatedSVD(5, random_state=SEED).fit(Xt)
Z = svd.transform(Xt)
_, Wt, Ht, _ = pynnmf.nmf_hals(Xt, 5, n=200, seed=SEED, init="nndsvd")
purity = lambda W: sum(max((labs[W.argmax(1) == k] == t).sum() for t in cat_names) for k in range(5)) / len(labs)  # noqa: E731
slide("Topics: five NMF topics and the LSI axes on 1000 BBC articles")
print(f"matrix {Xt.shape}; LSI singular values {svd.singular_values_.round(2)}")
for k in range(5):
    top = np.argsort(Ht[k])[::-1][:6]
    main = max(cat_names, key=lambda t: int((labs[Wt.argmax(1) == k] == t).sum()))
    print(f"NMF topic {k + 1} ({main}): {', '.join(vocab[top])}")
neg_mass = float(np.abs(np.minimum(svd.components_, 0)).sum() / np.abs(svd.components_).sum())
rand_pur = [purity(pynnmf.nmf_hals(Xt, 5, n=200, seed=s_, init="random")[1]) for s_ in range(5)]
print(f"negative mass of the LSI axes {neg_mass:.2f}, of the NMF topics 0")
print(f"topic purity: NNDSVD start {purity(Wt):.2f}; five random starts {min(rand_pur):.2f} to {max(rand_pur):.2f}")
assert Ht.min() >= 0 and neg_mass > 0.3 and purity(Wt) > max(rand_pur) + 0.1, "the start decides the topics"

rows = []
for q_, cat in (("goal referee", "sport"), ("election minister", "politics"), ("film oscar", "entertainment")):
    qv = vec.transform([q_]).toarray()
    shared = (Xt @ qv.T).ravel() > 0
    zq = svd.transform(qv)
    lat = (Z @ zq.T).ravel() / (np.linalg.norm(Z, axis=1) * np.linalg.norm(zq) + 1e-12)
    inc = labs == cat
    top = np.argsort(lat)[::-1][: int(inc.sum())]
    rows.append(
        {
            "query": q_,
            "articles sharing a word": int(shared.sum()),
            "of them in the category": int((shared & inc).sum()),
            "latent top 200 in the category": int(inc[top].sum()),
        }
    )
slide("Topics: queries in word space and in the latent space")
qt = pl.DataFrame(rows)
print(qt)
assert (qt["latent top 200 in the category"] > qt["of them in the category"] + 40).all()

f, ax = plt.subplots(1, 2, figsize=(plotlib.FULL, 1.3))
zq = svd.transform(vec.transform(["goal referee"]).toarray())
for t, col in zip(cat_names, (BLUE, ORANGE, GREEN, RED, PURPLE)):
    ax[0].scatter(Z[labs == t, 1], Z[labs == t, 2], color=col, s=3, label=t, alpha=0.6)
ax[0].scatter(zq[0, 1], zq[0, 2], color="black", marker="*", s=45, label="query")
ax[0].set_xlabel("LSI axis 2"), ax[0].set_ylabel("LSI axis 3")
ax[0].legend(fontsize=5.5, markerscale=2, ncol=3, loc="upper center", bbox_to_anchor=(0.5, -0.3))
order = np.argsort(labs, kind="stable")
ax[1].imshow(Wt[order].T / Wt.max(), cmap="viridis", aspect="auto")
ax[1].set_yticks(range(5)), ax[1].set_yticklabels([vocab[int(np.argmax(Ht[k]))] for k in range(5)], fontsize=6)
ax[1].set_xticks([100 + 200 * k for k in range(5)]), ax[1].set_xticklabels([t[:4] for t in cat_names], fontsize=6)
plotlib.save(f, "p7_topics")

## Slide: Initialisation — the problem is not convex

Eleven starts on four real problems, the same solver after each start. Families (Esposito, Mathematics 2021): random
(pyNNMF's `random`, and a draw scaled to the data); SVD based (pyNNMF `nndsvd` and `svd_impute`, and the completed-then-
NNDSVD of `faa.nmfinit`); clustering (k-means rows as parts); separable (successive projection); greedy rank-one
deflation; the sequential search (candidates proposed by the empty space algorithm or by a uniform draw, scored by the
residual they remove); and the **validated portfolio**: several of these starts get a short fit with some observed cells
hidden, and the start that predicts the hidden cells best is kept. Score: error on cells the fit never saw.

In [ ]:
Xall_h = Xall[[i for i, n in enumerate(names) if n.startswith("RH")]][:, :1440]
Xd, _, _ = data.load_table("digits", "target")
Xd = (Xd / Xd.max())[:600]
ones = np.ones(Xd.shape, dtype=bool)
miss = np.random.default_rng(SEED).random(Xd.shape) > 0.25
held_ts, ob_ts = ts_case(0)
PROBLEMS = {
    "ratings": dict(Xz=Rn, ob=train, tgt=test, truth=Rn, c=c_, solver="rwnmf", its=200),
    "temperature": dict(Xz=np.where(ob_ts, X, 0.0), ob=ob_ts, tgt=held_ts, truth=X, c=2, solver="hals", its=300),
    "humidity": dict(Xz=np.where(ob_ts, Xall_h, 0.0), ob=ob_ts, tgt=held_ts, truth=Xall_h, c=2, solver="hals", its=300),
    "digits 25% lost": dict(Xz=Xd, ob=miss, tgt=~miss, truth=Xd, c=8, solver="hals", its=100),
}
METHODS = [
    "pyNNMF random",
    "pyNNMF nndsvd",
    "pyNNMF svd_impute",
    "scaled random",
    "completed + NNDSVD",
    "k-means rows",
    "successive projection",
    "greedy rank-one",
    "sequential, uniform",
    "sequential, ESS",
    "validated portfolio",
]
STOCH = {
    "pyNNMF random",
    "scaled random",
    "k-means rows",
    "sequential, uniform",
    "sequential, ESS",
    "validated portfolio",
}


def make_fit(p: dict):
    def fit(ob: np.ndarray, W0: np.ndarray, H0: np.ndarray, rounds: int) -> np.ndarray:
        Xn = np.where(ob, p["Xz"], np.nan)
        init = lambda x, k: (W0.copy(), H0.copy())  # noqa: E731
        if p["solver"] == "hals":
            q = pynnmf.nmf_hals(Xn, p["c"], n=rounds, seed=0, init=init)
            return q[1] @ q[2]
        q = pynnmf.rwnmf(Xn, p["c"], alpha=best["alpha"], num_iter=rounds, seed=0, init=init)
        return q[1] @ q[2].T

    return fit


def start(name: str, pname: str, seed: int):
    p = PROBLEMS[pname]
    Xz_, ob_, c = p["Xz"], p["ob"], p["c"]
    Xn = np.where(ob_, Xz_, np.nan)
    if name == "pyNNMF random":
        return pynnmf.initialization.init_random(Xn, c, seed=seed)
    if name == "pyNNMF nndsvd":
        return pynnmf.initialization.init_nndsvd(Xn, c)
    if name == "pyNNMF svd_impute":
        return pynnmf.initialization.init_svd_impute(Xn, c)
    if name == "validated portfolio":
        vm = bss.ts_gaps(Xz_.shape, seed + 7, length=48, per_series=2) if pname in ("temperature", "humidity") else None
        return nmfinit.portfolio(Xz_, ob_, c, make_fit(p), seed=seed, val_mask=vm)[:2]
    fn = {
        "scaled random": nmfinit.random_scaled,
        "completed + NNDSVD": nmfinit.nndsvd_complete,
        "k-means rows": nmfinit.kmeans,
        "successive projection": nmfinit.spa,
        "greedy rank-one": nmfinit.greedy,
        "sequential, uniform": lambda *a: nmfinit.sequential(*a, use_ess=False),
        "sequential, ESS": lambda *a: nmfinit.sequential(*a, use_ess=True),
    }[name]
    return fn(Xz_, ob_, c, seed)


import time  # noqa: E402

t0 = time.time()
res = {}


def run_all(pname: str, methods=METHODS) -> None:
    p = PROBLEMS[pname]
    fit = make_fit(p)
    for m in methods:
        errs = []
        for s in range(3 if m in STOCH else 1):
            W0, H0 = start(m, pname, s)[:2]
            P = fit(p["ob"], W0, H0, p["its"])
            errs.append(float(np.sqrt(((P - p["truth"])[p["tgt"]] ** 2).mean())))
        res[(m, pname)] = (float(np.mean(errs)), float(np.std(errs)))


run_all("ratings", METHODS[:6])

In [ ]:
run_all("ratings", METHODS[6:])

In [ ]:
run_all("temperature")
run_all("humidity")
run_all("digits 25% lost")

In [ ]:
slide("Initialisation: eleven starts, four real problems (RMSE on unseen cells)")
tab = pl.DataFrame([{"start": m, **{p: res[(m, p)][0] for p in PROBLEMS}} for m in METHODS])
print(tab.with_columns(pl.col(pl.Float64).round(3)))
print(
    f"std over 3 seeds, pyNNMF random: { {p: round(res[('pyNNMF random', p)][1], 3) for p in PROBLEMS} }; time {time.time() - t0:.0f} s"
)
bestm = {p: min(METHODS, key=lambda m: res[(m, p)][0]) for p in PROBLEMS}
print("best start per problem:", bestm)
spread = {p: max(res[(m, p)][0] for m in METHODS) / min(res[(m, p)][0] for m in METHODS) - 1 for p in PROBLEMS}
print("worst / best - 1 per problem:", {p: round(v, 2) for p, v in spread.items()})

worst = {m: max(res[(m, p)][0] / min(res[(mm, p)][0] for mm in METHODS) for p in PROBLEMS) for m in METHODS}
print("worst case over the four problems (error / best error):", {m: round(v, 2) for m, v in worst.items()})
hum = PROBLEMS["humidity"]
Xn_h = np.where(hum["ob"], hum["Xz"], np.nan)
r1 = [
    float(
        np.sqrt(
            (
                (((lambda q: q[1] @ q[2])(pynnmf.nmf_hals(Xn_h, 1, n=300, seed=s_, init="random"))) - hum["truth"])[
                    hum["tgt"]
                ]
                ** 2
            ).mean()
        )
    )
    for s_ in range(3)
]
print(
    f"humidity at rank 1, three random starts: {np.mean(r1):.3f} +- {np.std(r1):.3f} (rank 2: best start {min(res[(m, 'humidity')][0] for m in METHODS):.3f}, worst {max(res[(m, 'humidity')][0] for m in METHODS):.3f})"
)
assert np.mean(r1) < 0.9 * min(res[(m, "humidity")][0] for m in METHODS) and np.std(r1) < 0.05, (
    "rank 1 is stable and better than any rank-2 start"
)
assert spread["humidity"] > 2 and spread["digits 25% lost"] < 0.06 and spread["ratings"] < 0.05, (
    "the start matters where the fit is not identifiable"
)
assert len(set(bestm.values())) >= 3, "no start wins everywhere"
assert worst["pyNNMF nndsvd"] < 1.05 < 1.3 < worst["validated portfolio"], (
    "nndsvd is the robust one; the validated portfolio is not"
)
assert all(
    abs(res[("sequential, ESS", p)][0] - res[("sequential, uniform", p)][0]) < 0.1 * res[("sequential, uniform", p)][0]
    for p in PROBLEMS
), "ESS proposals add nothing over uniform ones"

board = {(m, p): min(res[(mm, p)][0] for mm in METHODS) / res[(m, p)][0] for m in METHODS for p in PROBLEMS}
f = plotlib.score_board(board, METHODS, list(PROBLEMS), {p: BLUE for p in PROBLEMS}, left=0.3)
plotlib.save(f, "p7_init")